# IBM HR Attrition Analysis

## Notebook Data Contract

| Section | IN | Added | Functions available |
|---------|-----|-------|-------------------|
| 1 — EDA | raw CSV (1470 × 35) | `Attrition_bin` | `load_data()`, `inspect_data()` |
| 2 — Feature Engineering | Section 1 df (1470 × 33) | `income_band`, `age_group`, `is_manager`, `seniority_score`, `income_vs_dept_avg`, `overtime_bin` | + `drop_constants()`, `encode_target()`, `add_features()` |
| 3 — Modelling Baseline | `load_data()` called fresh — never inherits session state | fitted `pipe`, metrics | all of Section 2 |

> **Rule:** Every modelling cell calls `load_data()` at the top. No cell inherits `df` from a previous section.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

os.makedirs('figures', exist_ok=True)
os.makedirs('models', exist_ok=True)

sns.set_style('whitegrid')

In [ ]:
DATA = '/Users/kevinbarry/git/IBM_HR_Analytics/WA_Fn-UseC_-HR-Employee-Attrition.csv'

# Section 2 features are defined here — add_features() is the single source of truth
# for all engineered columns. Never add columns ad hoc in modelling cells.

def drop_constants(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18'], errors='ignore')

def encode_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['Attrition_bin'] = (df['Attrition'] == 'Yes').astype(int)
    return df

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    return df.assign(
        income_band=lambda x: pd.cut(
            x['MonthlyIncome'],
            bins=[0, 4000, 8000, 20_000],
            labels=['Low', 'Mid', 'High']
        ),
        age_group=lambda x: pd.cut(
            x['Age'],
            bins=[18, 30, 40, 50, 65],
            labels=['20s', '30s', '40s', '50s+']
        ),
        is_manager=lambda x: x['JobRole'].str.contains('Manager').astype(int),
        seniority_score=lambda x: x['YearsAtCompany'] * x['JobLevel'],
        income_vs_dept_avg=lambda x: (
            x['MonthlyIncome']
            - x.groupby('Department')['MonthlyIncome'].transform('mean')
        ),
        overtime_bin=lambda x: (x['OverTime'] == 'Yes').astype(int)
    )

def load_data(path: str) -> pd.DataFrame:
    return (
        pd.read_csv(path)
          .pipe(drop_constants)
          .pipe(encode_target)
          .pipe(add_features)
    )

df = load_data(DATA)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

---
## Section 1 — EDA

**IN:** raw CSV → `load_data()` → df (1470 × 39)  
**Goal:** Understand distributions, class balance, key relationships before any modelling.

In [ ]:
def inspect_data(df: pd.DataFrame) -> None:
    print(f'Shape: {df.shape}')
    print(f'\nAttrition split:\n{df["Attrition_bin"].value_counts(normalize=True).round(3)}')
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f'\nNulls: {"None" if len(nulls) == 0 else nulls.to_string()}')
    summary = (
        df.groupby('Department')
          .agg(
              headcount=('EmployeeNumber', 'count'),
              avg_income=('MonthlyIncome', 'mean'),
              attrition_rate=('Attrition_bin', 'mean')
          )
          .round(2)
          .reset_index()
    )
    print(f'\nDept summary:\n{summary.to_string()}')

inspect_data(df)

### Dataset: IBM HR Employee Attrition

| | |
|---|---|
| **Shape** | (1470, 39) |
| **Target** | `Attrition_bin` — 16.1% positive (imbalanced 5:1) |
| **Key numerics** | MonthlyIncome, Age, YearsAtCompany, JobLevel, TotalWorkingYears |
| **Dropped constants** | EmployeeCount, StandardHours, Over18 |
| **Nulls** | None |
| **Engineered** | income_band, age_group, is_manager, seniority_score, income_vs_dept_avg, overtime_bin |

**Implication of class imbalance:** Accuracy is a misleading metric — a model predicting "Stay" for everyone achieves 84%. Use F1 on the minority class.

In [ ]:
print(df.describe())

In [ ]:
dept_summary = (
    df.groupby('Department')
      .agg(
          headcount=('EmployeeNumber', 'count'),
          avg_income=('MonthlyIncome', 'mean'),
          attrition_rate=('Attrition_bin', 'mean'),
          avg_age=('Age', 'mean')
      )
      .round(2)
      .reset_index()
)
print(dept_summary.to_string())

In [ ]:
# Income by Department × Attrition — do leavers earn more or less than stayers?
income_pivot = pd.pivot_table(
    df,
    values='MonthlyIncome',
    index='Department',
    columns='Attrition',
    aggfunc='mean',
    fill_value=0
).round(0)

income_gap = income_pivot['Yes'] - income_pivot['No']
print(income_pivot)
print(f'\nIncome gap (leaver minus stayer):\n{income_gap.sort_values()}')

**Finding:** Leavers earn less than stayers across all three departments — income gap ranges from ~$1,300 in Sales to ~$3,600 in HR.

**Implication:** Lower-paid employees are disproportionately leaving. Compensation may be a retention lever, particularly in HR where the gap is largest.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(
    dept_summary['Department'],
    dept_summary['attrition_rate'],
    color='#39B798',
    edgecolor='white',
    linewidth=0.8,
    width=0.5
)

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f"{bar.get_height():.1%}",
        ha='center', va='bottom',
        fontsize=10, fontweight='bold', color='#333333'
    )

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_ylim(0, dept_summary['attrition_rate'].max() * 1.25)
ax.set_title('Attrition Rate by Department', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('Attrition Rate', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figures/attrition_by_department.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Sales has the highest attrition rate (21%), HR is second (19%), R&D the lowest (14%) despite having the largest headcount.

**Implication:** Sales is the highest-risk department. Volume of leavers is concentrated in R&D (133 of 237 total) but rate is worst in Sales — two different retention problems.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# [0,0] Attrition count by Department
sns.countplot(x='Department', hue='Attrition', data=df, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Attrition Count by Department')
axes[0, 0].set_xlabel('')

# [0,1] Monthly Income by Attrition
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df, ax=axes[0, 1],
            palette='Set1', hue='Attrition', legend=False)
axes[0, 1].set_title('Monthly Income by Attrition')

# [1,0] Age distribution by Attrition
axes[1, 0].hist(df[df['Attrition'] == 'Yes']['Age'], bins=20, alpha=0.6, label='Yes', color='tomato')
axes[1, 0].hist(df[df['Attrition'] == 'No']['Age'],  bins=20, alpha=0.4, label='No',  color='steelblue')
axes[1, 0].set_title('Age Distribution by Attrition')
axes[1, 0].set_xlabel('Age')
axes[1, 0].legend()

# [1,1] Tenure by Attrition
sns.violinplot(x='Attrition', y='YearsAtCompany', data=df, ax=axes[1, 1],
               palette='muted', hue='Attrition', legend=False)
axes[1, 1].set_title('Tenure by Attrition')

plt.suptitle('IBM HR Attrition — EDA Overview', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('figures/eda_2x2.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Attrition is concentrated in employees aged 25–35 with under 5 years tenure and lower income. The violin plot shows leavers cluster at near-zero YearsAtCompany.

**Implication:** The first 1–2 years is the critical retention window. A high-risk profile emerges: early-career, lower-paid, Sales department. These three factors in combination are the primary modelling signal.

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c != 'EmployeeNumber']

corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Matrix — IBM HR (lower triangle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

attrition_corr = (
    corr['Attrition_bin']
      .drop('Attrition_bin')
      .abs()
      .sort_values(ascending=False)
)
print('Top 10 features correlated with Attrition_bin:\n', attrition_corr.head(10))

**Finding:** JobLevel–MonthlyIncome correlation is 0.95 — seniority drives pay almost entirely. No single feature has a strong linear correlation with `Attrition_bin` (top correlates are typically overtime_bin, JobLevel, MonthlyIncome).

**Implication:** Attrition has no single dominant predictor — it's driven by combinations of features. A linear model will underfit; tree-based models (Week 3) should improve on the baseline.

---
## Section 2 — Feature Engineering

**IN:** Section 1 df (1470 × 33)  
**OUT:** df (1470 × 39) — all engineered features produced by `add_features()` in `load_data()`  
**Rule:** No ad hoc column creation below this point. If a new feature is needed, add it to `add_features()` and reload.

| Feature | Construction | Rationale |
|---------|-------------|-----------|
| `income_band` | `pd.cut` on MonthlyIncome — Low/Mid/High | Captures non-linear income effects |
| `age_group` | `pd.cut` on Age — 20s/30s/40s/50s+ | Age as lifecycle stage, not continuous |
| `is_manager` | JobRole contains 'Manager' → 0/1 | Seniority proxy independent of JobLevel |
| `seniority_score` | YearsAtCompany × JobLevel | Combined tenure + level signal |
| `income_vs_dept_avg` | MonthlyIncome − dept mean (via transform) | Relative pay within peer group |
| `overtime_bin` | OverTime == 'Yes' → 0/1 | Encodes string flag as numeric |

In [ ]:
# Verify all engineered features are present and correctly typed
feature_check = df[['income_band', 'age_group', 'is_manager',
                     'seniority_score', 'income_vs_dept_avg', 'overtime_bin']]

print(feature_check.dtypes)
print()
print(feature_check.head(5).to_string())
print()
print('income_band counts:\n', df['income_band'].value_counts().sort_index())
print()
print('age_group counts:\n', df['age_group'].value_counts().sort_index())
print()
print('NaNs in engineered features:\n', feature_check.isnull().sum())

---
## Section 3 — Modelling Baseline

**IN:** `load_data()` called fresh at the top of each modelling cell  
**Goal:** Establish baseline metrics — LinearRegression warm-up, then LogisticRegression classification baseline with full ColumnTransformer Pipeline.  
**Primary metric:** F1 on the Leave class (not accuracy — class imbalance makes accuracy misleading).

In [ ]:
# Regression warm-up: predict MonthlyIncome
# Purpose: confirm Pipeline mechanics before tackling classification

df = load_data(DATA)

features = ['Age', 'JobLevel', 'TotalWorkingYears', 'YearsAtCompany',
            'YearsInCurrentRole', 'YearsSinceLastPromotion']

X = df[features]
y = df['MonthlyIncome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on train only
X_test_sc  = scaler.transform(X_test)        # transform only — no fit on test

lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred = lr.predict(X_test_sc)

print(f'R²:  {r2_score(y_test, y_pred):.3f}')
print(f'MAE: ${mean_absolute_error(y_test, y_pred):,.0f}')

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.4, s=20)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('Actual Monthly Income')
ax.set_ylabel('Predicted')
ax.set_title(f'LinearRegression — Predicted vs Actual  (R²={r2_score(y_test, y_pred):.3f})')
plt.tight_layout()
plt.show()

**Finding:** R² ~0.895. The predicted vs actual plot shows clear horizontal banding — income clusters at job level thresholds. JobLevel dominates predictions.

**Implication:** The model is largely learning income by job level. Continuous features add marginal signal. This is expected and not a problem — it confirms JobLevel is the dominant income driver.

In [ ]:
# Cross-validate the same model inside a Pipeline
# Pipeline prevents data leakage — scaler is fit only on training folds

df = load_data(DATA)

features = ['Age', 'JobLevel', 'TotalWorkingYears', 'YearsAtCompany',
            'YearsInCurrentRole', 'YearsSinceLastPromotion']

X = df[features]
y = df['MonthlyIncome']

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')

print(f'CV R² scores: {cv_scores.round(3)}')
print(f'Mean: {cv_scores.mean():.3f}  ±  {cv_scores.std():.3f}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)
single_r2 = r2_score(y_test, pipe.predict(X_test))

print(f'\nSingle split R²: {single_r2:.3f}')
print(f'CV mean R²:      {cv_scores.mean():.3f}')
print(f'Difference:      {abs(single_r2 - cv_scores.mean()):.3f}')

**Finding:** CV mean R² 0.904 ± 0.011 — tight spread across all 5 folds. Single split delta is 0.009.

**Implication:** Model is stable — no fold produces a meaningfully worse score. Low variance confirms this isn't a lucky split.

In [ ]:
# Classification baseline: predict Attrition_bin
# ColumnTransformer handles numeric scaling, categorical OHE, and binary passthrough
# in a single leakage-safe Pipeline

df = load_data(DATA)

num_cols = ['Age', 'MonthlyIncome', 'JobLevel', 'TotalWorkingYears',
            'YearsAtCompany', 'YearsInCurrentRole', 'seniority_score', 'income_vs_dept_avg']
cat_cols = ['Department', 'JobRole', 'MaritalStatus']
bin_cols = ['overtime_bin', 'is_manager']   # already 0/1 — passthrough

preprocessor = ColumnTransformer(transformers=[
    ('num',  StandardScaler(),                                             num_cols),
    ('cat',  OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('pass', 'passthrough',                                                bin_cols)
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

X = df[num_cols + cat_cols + bin_cols]
y = df['Attrition_bin']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipe.fit(X_train, y_train)

print(f'Transformed feature count: {pipe.named_steps["preprocessor"].transform(X_train).shape[1]}')
print()
print(classification_report(y_test, pipe.predict(X_test), target_names=['Stay', 'Leave']))

cv_f1 = cross_val_score(pipe, X, y, cv=5, scoring='f1')
print(f'CV F1: {cv_f1.mean():.3f}  ±  {cv_f1.std():.3f}')

**Finding:** CV F1 ~0.27 on the Leave class. The model catches very few actual leavers (low recall) — it defaults toward predicting Stay due to class imbalance.

**Implication:** LogisticRegression on imbalanced data without class weighting is the expected weak baseline. This is the number to beat in Section 4 with RandomForest + `class_weight='balanced'`. The ColumnTransformer Pipeline is correctly structured — the issue is the model, not the preprocessing.